# Juiz de Bocha — treino completo (pipeline original)

Reproduz no Colab o treino do repositório [juiz-de-bocha-algoritmo](https://github.com/JonathasmpDEV/juiz-de-bocha-algoritmo):

| Etapa | Script original | O que faz |
|---|---|---|
| 1 | `train/prepare_datasets.py` | Subconjunto **sports ball** do COCO 2017 |
| 2 | `train/train.py` | Treina Mask R-CNN R50 só com "sports ball" → `weights_first_train` |
| 3 | `train_custom_dataset/2_generate_labels_files.py` | Usa o modelo da etapa 2 para **rotular automaticamente** as fotos da pasta `bocha` |
| 4 | `train_custom_dataset/fix_deleted_dataset.py` | Revisão: você descarta as fotos mal rotuladas |
| 5 | `train_custom_dataset/3_train.py` | Ajuste fino nas fotos de bocha com aumentos de dados |
| 6 | `train_custom_dataset/4_cases_validation.py` | Valida com a lógica do juiz (confiança 0.82, como em produção) e exporta o modelo |

**Antes de rodar:** *Ambiente de execução → Alterar tipo → GPU (T4)*. Tudo que importa (checkpoints, rótulos, modelo final) é salvo no Drive em `00-Juiz de Bocha/treino`; se o Colab desconectar, rode as células de novo que o treino **continua de onde parou**.

**Adaptações ao Colab** (o resto é igual ao original):
- Etapa 1 baixa o COCO 2017 completo, como o original (dá para trocar por um download reduzido com `COCO_COMPLETO = False`).
- Checkpoints a cada 1000 iterações, mantendo só os 2 últimos.
- Corrigido um bug do `2_generate_labels_files.py`: os polígonos usavam sempre `contours[0]`.

## 0. Ambiente

In [ ]:
!nvidia-smi -L || echo "Sem GPU — ative em Ambiente de execução → Alterar tipo"
try:
    import detectron2
    print("detectron2 já instalado:", detectron2.__version__)
except ImportError:
    !pip install -q exifread
    !pip install -q --no-build-isolation 'git+https://github.com/facebookresearch/detectron2.git'
    import detectron2
    print("detectron2 instalado:", detectron2.__version__)

In [ ]:
import os, sys
from google.colab import drive
drive.mount("/content/drive")

REPO_DIR = "/content/juiz-de-bocha-algoritmo"
if not os.path.isdir(REPO_DIR):
    !git clone -q https://github.com/JonathasmpDEV/juiz-de-bocha-algoritmo.git {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -1
sys.path.insert(0, REPO_DIR)

FOTOS_BOCHA = "/content/drive/MyDrive/BACKUP PENDRIVE 256/CursoSegImgPython/bocha"  # @param {type:"string"}
SAIDA = "/content/drive/MyDrive/00-Juiz de Bocha/treino"  # @param {type:"string"}
ETAPA1_DIR = f"{SAIDA}/etapa1_coco_sports_ball"
ETAPA2_DIR = f"{SAIDA}/etapa2_bocha"
MODELO_FINAL_DIR = f"{SAIDA}/modelo_final"
for d in (ETAPA1_DIR, ETAPA2_DIR, MODELO_FINAL_DIR):
    os.makedirs(d, exist_ok=True)
print("Fotos de bocha:", len([f for f in os.listdir(FOTOS_BOCHA) if not f.startswith("._")]))

In [ ]:
# Trainer comum às duas etapas: avaliação COCO e checkpoints limitados (para não lotar o Drive)
from detectron2.engine import DefaultTrainer, hooks
from detectron2.evaluation import COCOEvaluator, DatasetEvaluators

class Trainer(DefaultTrainer):
    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        return DatasetEvaluators([COCOEvaluator(dataset_name, output_dir=output_folder or cfg.OUTPUT_DIR)])

    def build_hooks(self):
        hs = super().build_hooks()
        for i, h in enumerate(hs):
            if isinstance(h, hooks.PeriodicCheckpointer):
                hs[i] = hooks.PeriodicCheckpointer(self.checkpointer, self.cfg.SOLVER.CHECKPOINT_PERIOD, max_to_keep=2)
        return hs

## 1. Dataset COCO "sports ball" (`train/prepare_datasets.py`)
Com `COCO_COMPLETO = True` (padrão, igual ao original) baixa o COCO 2017 inteiro: 118 mil imagens de treino (~18 GB) + validação (~1 GB), leva ~30–40 min. Com `False`, baixa só as imagens com bola + `NEGATIVOS_TREINO` imagens sem bola. Em ambos os casos filtra a categoria *sports ball* (renumerada para id 1) e, como no original, a validação fica só com imagens que têm bola.

> **Atenção:** o Detectron2 descarta no treino as imagens sem nenhuma anotação (`DATALOADER.FILTER_EMPTY_ANNOTATIONS = True`, padrão — o original também não mudava isso). Na prática o modelo treina só com as imagens que têm bola nas duas opções; o download completo só reproduz o original ao pé da letra.

In [ ]:
import json, random, zipfile, urllib.request
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

COCO_DIR = "/content/coco"
COCO_COMPLETO = True  # @param {type:"boolean"}
NEGATIVOS_TREINO = 3000  # @param {type:"integer"}
# NEGATIVOS_TREINO só vale com COCO_COMPLETO = False

os.makedirs(f"{COCO_DIR}/annotations", exist_ok=True)
if not os.path.exists(f"{COCO_DIR}/annotations/instances_train2017.json"):
    urllib.request.urlretrieve("http://images.cocodataset.org/annotations/annotations_trainval2017.zip", "/content/ann.zip")
    with zipfile.ZipFile("/content/ann.zip") as z:
        for n in ("annotations/instances_train2017.json", "annotations/instances_val2017.json"):
            z.extract(n, COCO_DIR)
    os.remove("/content/ann.zip")

def gerar_dados(fase):
    with open(f"{COCO_DIR}/annotations/instances_{fase}2017.json") as f:
        coco = json.load(f)
    cat = next(c for c in coco["categories"] if c["name"] == "sports ball")
    ball_id, cat = cat["id"], {**cat, "id": 1}
    anns = [{**a, "category_id": 1} for a in coco["annotations"] if a["category_id"] == ball_id]
    com_bola = {a["image_id"] for a in anns}
    imagens = [im for im in coco["images"] if im["id"] in com_bola]
    if fase == "train" and COCO_COMPLETO:
        imagens = coco["images"]  # original: treino com todas as imagens
    elif fase == "train":
        sem_bola = [im for im in coco["images"] if im["id"] not in com_bola]
        random.seed(2)
        imagens += random.sample(sem_bola, min(NEGATIVOS_TREINO, len(sem_bola)))
    return dict(images=imagens, annotations=anns, categories=[cat])

def baixar(args):
    fase, nome = args
    destino = f"{COCO_DIR}/{fase}2017/{nome}"
    if not os.path.exists(destino):
        urllib.request.urlretrieve(f"http://images.cocodataset.org/{fase}2017/{nome}", destino)

def baixar_zip(fase):
    pasta = f"{COCO_DIR}/{fase}2017"
    if os.path.isdir(pasta) and len(os.listdir(pasta)) > (100000 if fase == "train" else 4000):
        return
    !wget -c -q --show-progress http://images.cocodataset.org/zips/{fase}2017.zip -O /content/{fase}2017.zip
    !unzip -q -o /content/{fase}2017.zip -d {COCO_DIR}
    !rm /content/{fase}2017.zip

for fase in ("train", "val"):
    if COCO_COMPLETO:
        baixar_zip(fase)
    dados = gerar_dados(fase)
    with open(f"{COCO_DIR}/annotations/juiz_de_bocha_{fase}.json", "w") as f:
        json.dump(dados, f)
    os.makedirs(f"{COCO_DIR}/{fase}2017", exist_ok=True)
    with ThreadPoolExecutor(32) as ex:
        list(tqdm(ex.map(baixar, [(fase, im["file_name"]) for im in dados["images"]]),
                  total=len(dados["images"]), desc=f"{fase}2017"))
    print(fase, "→", len(dados["images"]), "imagens,", len(dados["annotations"]), "bolas anotadas")

## 2. Treino etapa 1 — COCO "sports ball" (`train/train.py`)
Mesmos hiperparâmetros do original: parte dos pesos COCO R50, LR 0.005, batch 2, **15000 iterações** (~1,5–2 h numa T4). Gera o `weights_first_train`.

In [ ]:
from detectron2.config import get_cfg
from detectron2.data.datasets import register_coco_instances
from detectron2.data import DatasetCatalog

def registrar(nome, json_file, image_root):
    if nome in DatasetCatalog.list():
        DatasetCatalog.remove(nome)
        from detectron2.data import MetadataCatalog
        MetadataCatalog.remove(nome)
    register_coco_instances(nome, {"thing_classes": ["sports ball"]}, json_file, image_root)

registrar("juiz_de_bocha_train", f"{COCO_DIR}/annotations/juiz_de_bocha_train.json", f"{COCO_DIR}/train2017")
registrar("juiz_de_bocha_val", f"{COCO_DIR}/annotations/juiz_de_bocha_val.json", f"{COCO_DIR}/val2017")

ITERACOES_ETAPA1 = 15000  # @param {type:"integer"}

cfg1 = get_cfg()
cfg1.merge_from_file(f"{REPO_DIR}/models/mask_rcnn_juiz_de_bocha/mask_rcnn_juiz_de_bocha.yaml")
cfg1.MODEL.WEIGHTS = "detectron2://COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x/137849600/model_final_f10217.pkl"
cfg1.DATALOADER.NUM_WORKERS = 2
cfg1.SOLVER.IMS_PER_BATCH = 2
cfg1.SOLVER.BASE_LR = 0.005
cfg1.SOLVER.MAX_ITER = ITERACOES_ETAPA1
cfg1.SOLVER.CHECKPOINT_PERIOD = 1000
cfg1.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 128
cfg1.OUTPUT_DIR = ETAPA1_DIR

trainer = Trainer(cfg1)
trainer.resume_or_load(resume=True)  # continua do último checkpoint no Drive
trainer.train()
WEIGHTS_FIRST_TRAIN = f"{ETAPA1_DIR}/model_final.pth"
print("Etapa 1 concluída:", WEIGHTS_FIRST_TRAIN)

## 3. Rotulagem automática das fotos de bocha (`2_generate_labels_files.py`)
O modelo da etapa 1 (confiança 0.7) gera as máscaras de cada foto da pasta `bocha`. Divisão 80/20 treino/validação com `random.seed(2)`, como no original.

In [ ]:
import shutil
import numpy as np
from pycocotools import mask as mask_tool
from skimage import measure
from recognizer import Recognizer

BOCHA_DIR = "/content/bocha"
DATA_DIR = f"{BOCHA_DIR}/data"
os.makedirs(DATA_DIR, exist_ok=True)
for nome in sorted(os.listdir(FOTOS_BOCHA)):
    if nome.startswith("._") or not nome.lower().endswith((".jpg", ".jpeg", ".png")):
        continue
    if not os.path.exists(f"{DATA_DIR}/{nome}"):
        shutil.copy(f"{FOTOS_BOCHA}/{nome}", f"{DATA_DIR}/{nome}")

rotulador = Recognizer(
    config_file=f"{REPO_DIR}/models/mask_rcnn_juiz_de_bocha/mask_rcnn_juiz_de_bocha.yaml",
    weights_file=f"{ETAPA1_DIR}/model_final.pth",
    confidence_threshold=0.7,
    gpu_id=0,
)

amostras = list(enumerate(sorted(os.listdir(DATA_DIR))))
random.seed(2)
random.shuffle(amostras)
corte = int(0.8 * len(amostras))
divisao = {"train": amostras[:corte], "val": amostras[corte:]}

rotulos = {}  # image_id -> {imagem, anotacoes}
ann_id = 1
for fase, lista in divisao.items():
    for i, nome in tqdm(lista, desc=fase):
        im = rotulador.read_file(f"{DATA_DIR}/{nome}")
        anots = []
        for m in rotulador.predict_instances(im).get("pred_masks"):
            m = np.asfortranarray(np.array(m, dtype=np.uint8))
            rle = mask_tool.encode(m)
            contornos = measure.find_contours(m, level=0.5)
            seg = [np.flip(c, axis=1).ravel().tolist() for c in contornos if len(c) >= 3]
            if not seg:
                continue
            anots.append(dict(image_id=i, category_id=1, id=ann_id, iscrowd=0,
                              area=float(mask_tool.area(rle)), bbox=mask_tool.toBbox(rle).tolist(),
                              segmentation=seg))
            ann_id += 1
        rotulos[i] = dict(fase=fase, imagem=dict(id=i, file_name=nome, height=im.shape[0], width=im.shape[1]),
                          anotacoes=anots)

sem_bola = [r["imagem"]["file_name"] for r in rotulos.values() if not r["anotacoes"]]
print(f"{len(rotulos)} fotos rotuladas | {sum(len(r['anotacoes']) for r in rotulos.values())} bolas | {len(sem_bola)} sem nenhuma bola")

## 4. Revisão dos rótulos (`fix_deleted_dataset.py`)
Confira a grade: cada foto mostra o número (`image_id`) e as máscaras geradas. Coloque em `EXCLUIR` os números das fotos mal rotuladas (bola faltando, sujeira/objeto marcado como bola, balim estranho) e rode a célula seguinte.

In [ ]:
import matplotlib.pyplot as plt
from detectron2.utils.visualizer import Visualizer

def desenhar(r):
    im = Recognizer.read_file(f"{DATA_DIR}/{r['imagem']['file_name']}")
    v = Visualizer(im, scale=0.35)
    for a in r["anotacoes"]:
        v.draw_polygon(np.array(a["segmentation"][0]).reshape(-1, 2) * 1.0, color="lime", alpha=0.35)
    return v.output.get_image()

ids = sorted(rotulos)
COLUNAS = 4
for ini in range(0, len(ids), 16):
    bloco = ids[ini:ini + 16]
    fig, axs = plt.subplots((len(bloco) + COLUNAS - 1) // COLUNAS, COLUNAS, figsize=(18, 4.5 * ((len(bloco) + COLUNAS - 1) // COLUNAS)))
    for ax in np.ravel(axs):
        ax.axis("off")
    for ax, i in zip(np.ravel(axs), bloco):
        r = rotulos[i]
        ax.imshow(desenhar(r))
        ax.set_title(f"#{i} [{r['fase']}] {len(r['anotacoes'])} bolas", fontsize=11)
    plt.tight_layout(); plt.show()

In [ ]:
EXCLUIR = []  # @param {type:"raw"}
# ex.: EXCLUIR = [3, 17, 42]

def salvar_fase(fase):
    base = {"info": {"description": "Juiz de Bocha", "version": "2.0"},
            "licenses": [], "categories": [{"id": 1, "name": "sports ball", "supercategory": "sports"}],
            "images": [], "annotations": []}
    for i, r in rotulos.items():
        if r["fase"] != fase or i in EXCLUIR:
            continue
        base["images"].append(r["imagem"])
        base["annotations"].extend(r["anotacoes"])
    caminho = f"{SAIDA}/juiz_de_bocha_{fase}.json"  # fica salvo no Drive
    with open(caminho, "w") as f:
        json.dump(base, f)
    print(f"{fase}: {len(base['images'])} fotos, {len(base['annotations'])} bolas → {caminho}")
    return caminho

json_train, json_val = salvar_fase("train"), salvar_fase("val")
registrar("juiz_de_bocha_train", json_train, DATA_DIR)
registrar("juiz_de_bocha_val", json_val, DATA_DIR)

## 5. Treino etapa 2 — fotos de bocha (`3_train.py`)
Parte do `weights_first_train`. LR 0.0025, batch 3, 1000 iterações, com os mesmos aumentos de dados do original (brilho, flip, crop 640, rotação ±90°, contraste, iluminação, saturação).

In [ ]:
from detectron2.data import DatasetMapper, build_detection_train_loader
from detectron2.data import transforms as T

ITERACOES_ETAPA2 = 1000  # @param {type:"integer"}

class TrainerBocha(Trainer):
    @classmethod
    def build_train_loader(cls, cfg):
        return build_detection_train_loader(cfg, mapper=DatasetMapper(cfg, is_train=True, augmentations=[
            T.RandomBrightness(0.5, 1.5),
            T.RandomFlip(prob=0.5),
            T.RandomCrop("absolute_range", (640, 640)),
            T.RandomRotation(angle=[-90, 90], expand=True),
            T.RandomContrast(intensity_min=0.2, intensity_max=1.8),
            T.RandomLighting(scale=1.0),
            T.RandomSaturation(intensity_min=0.2, intensity_max=1.8),
        ]))

cfg2 = get_cfg()
cfg2.merge_from_file(f"{REPO_DIR}/models/mask_rcnn_juiz_de_bocha_custom/mask_rcnn_juiz_de_bocha.yaml")
cfg2.MODEL.WEIGHTS = f"{ETAPA1_DIR}/model_final.pth"
cfg2.DATALOADER.NUM_WORKERS = 2
cfg2.SOLVER.IMS_PER_BATCH = 3
cfg2.SOLVER.BASE_LR = 0.0025
cfg2.SOLVER.MAX_ITER = ITERACOES_ETAPA2
cfg2.SOLVER.CHECKPOINT_PERIOD = 250
cfg2.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 128
cfg2.OUTPUT_DIR = ETAPA2_DIR

trainer = TrainerBocha(cfg2)
trainer.resume_or_load(resume=True)
trainer.train()
print("Etapa 2 concluída:", f"{ETAPA2_DIR}/model_final.pth")

In [ ]:
# Curvas de perda (equivalente ao train/plot.py)
import pandas as pd
for nome, d in (("Etapa 1", ETAPA1_DIR), ("Etapa 2", ETAPA2_DIR)):
    m = pd.read_json(f"{d}/metrics.json", lines=True)
    m = m[m["total_loss"].notna()] if "total_loss" in m else m
    cols = [c for c in ("total_loss", "loss_cls", "loss_box_reg", "loss_mask") if c in m]
    m.plot(x="iteration", y=cols, figsize=(12, 3.5), title=nome, logy=True); plt.show()

## 6. Validação com a lógica do juiz (`4_cases_validation.py`) e exportação
Usa o modelo final com confiança **0.82** (valor de produção em `cloud-run-function/main.py`) nas fotos de validação.

In [ ]:
import math, cv2

def processar_bolas(img, instancias):
    instancias = [i for i in instancias if i["class"] == "sports ball"]
    bolas, balim, vencedora = [], None, None
    if len(instancias) >= 2:
        margem = 0.10
        x_min, x_max = img.shape[1] * margem, img.shape[1] * (1 - margem)
        y_min, y_max = img.shape[0] * margem, img.shape[0] * (1 - margem)
        no_meio, nas_bordas = [], []
        for inst in instancias:
            c = inst["center"]
            (no_meio if x_min <= c["x"] <= x_max and y_min <= c["y"] <= y_max else nas_bordas).append(inst)
        if no_meio:
            no_meio.sort(key=lambda i: i["area"])
            while True:
                balim = no_meio.pop(0)
                if not no_meio:
                    break
                if math.fabs(no_meio[-1]["area"] / balim["area"] - no_meio[0]["area"] / balim["area"]) <= 1.5:
                    break
            nas_bordas = [b for b in nas_bordas if b["area"] >= balim["area"] / 2]
            bolas = no_meio + nas_bordas
            for b in bolas:
                b["distance"] = math.hypot(b["center"]["x"] - balim["center"]["x"], b["center"]["y"] - balim["center"]["y"])
            bolas.sort(key=lambda b: b["distance"])
            vencedora = bolas[0] if bolas else None
    return bolas, vencedora, balim

juiz = Recognizer(
    config_file=f"{REPO_DIR}/models/mask_rcnn_juiz_de_bocha_custom/mask_rcnn_juiz_de_bocha.yaml",
    weights_file=f"{ETAPA2_DIR}/model_final.pth",
    confidence_threshold=0.82,
    gpu_id=0,
)

val_ids = [i for i, r in rotulos.items() if r["fase"] == "val" and i not in EXCLUIR]
fig, axs = plt.subplots((len(val_ids) + 3) // 4, 4, figsize=(18, 4.5 * ((len(val_ids) + 3) // 4)))
for ax in np.ravel(axs):
    ax.axis("off")
decididas = 0
for ax, i in zip(np.ravel(axs), val_ids):
    img = Recognizer.read_file(f"{DATA_DIR}/{rotulos[i]['imagem']['file_name']}")
    bolas, vencedora, balim = processar_bolas(img, juiz.predict_data(img))
    out = img.copy()
    esp = max(3, int(max(img.shape[:2]) * 0.006))
    def circulo(inst, cor, extra=0):
        b = inst["box"]
        cv2.ellipse(out, (int(inst["center"]["x"]), int(inst["center"]["y"])),
                    (int((b["x2"] - b["x1"]) / 2) + extra, int((b["y2"] - b["y1"]) / 2) + extra), 0, 0, 360, cor, esp)
    for b in bolas[1:]:
        circulo(b, (255, 200, 0))
    if balim:
        circulo(balim, (255, 255, 255)); decididas += 1
    if vencedora:
        circulo(vencedora, (50, 230, 50), extra=10)
    ax.imshow(out); ax.set_title(f"#{i} {'ok' if balim else 'sem decisão'} | {len(bolas)} bolas", fontsize=11)
plt.tight_layout(); plt.show()
print(f"Juiz decidiu em {decididas}/{len(val_ids)} fotos de validação (branco = balim, verde = vencedora)")

In [ ]:
# Exporta no formato que o cloud-run-function/main.py espera (models/model/config.yaml + model_final.pth)
cfg_export = cfg2.clone()
cfg_export.MODEL.WEIGHTS = "models/model/model_final.pth"
with open(f"{MODELO_FINAL_DIR}/config.yaml", "w") as f:
    f.write(cfg_export.dump())
shutil.copy(f"{ETAPA2_DIR}/model_final.pth", f"{MODELO_FINAL_DIR}/model_final.pth")
!ls -lh "{MODELO_FINAL_DIR}"